# Meth3D-Net V6 — TCGA Probe Extractor v1
## Complete pipeline: download files + extract DMB-overlapping probes

### How file download works
The 100 V6 DMB files are stored as **Google Sheets** in Drive (uploaded as CSV → auto-converted).
This notebook uses `requests` with your Colab auth token — the simplest method that works
with authenticated Drive files without needing the Drive API or gdown.

### Steps
1. Mount Drive and set paths
2. Download all 100 V6 DMB files using authenticated requests
3. Verify the download
4. Load V6 DMBs (all chromosomes)
5. Build probe set from Illumina 450K manifest
6. Scan TCGA methylation file (~20–60 min)
7. Write and verify output

**Required in `MyDrive/Meth3DNet_TCGA/`:**
- `jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv` (~49 GB)
- `humanmethylation450_15017482_v1-2.csv` (Illumina manifest)
- `Methylation_Paper_CpG_v6/` folder (already has chr20-22/X/Y files)


## Step 0 — Mount Drive and configure paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH   = '/content/drive/MyDrive/Meth3DNet_TCGA'
DEST_SUB    = f'{BASE_PATH}/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'
METH_FILE   = f'{BASE_PATH}/jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv'
MANIFEST    = f'{BASE_PATH}/humanmethylation450_15017482_v1-2.csv'
OUTPUT_FILE = f'{BASE_PATH}/tcga_dmb_probes_full.tsv'
WINDOW_SIZE = 50_000

os.makedirs(DEST_SUB, exist_ok=True)

print('Paths configured:')
for name, path in [('Base', BASE_PATH), ('DMB subfolder', DEST_SUB),
                    ('TCGA file', METH_FILE), ('Manifest', MANIFEST),
                    ('Output', OUTPUT_FILE)]:
    exists = os.path.exists(path)
    sz = ''
    if exists and os.path.isfile(path):
        gb = os.path.getsize(path)/1e9
        sz = f'  ({gb:.1f} GB)' if gb > 0.5 else f'  ({os.path.getsize(path)/1e6:.0f} MB)'
    print(f'  {"OK" if exists else "MISSING"}: {name}{sz}')


Mounted at /content/drive
Paths configured:
  OK: Base
  OK: DMB subfolder
  OK: TCGA file  (49.2 GB)
  OK: Manifest  (197 MB)
  OK: Output  (12.8 GB)


## Step 1 — Download 100 V6 DMB files
Uses authenticated `requests` session — works with your logged-in Google account.
Files stored as Google Sheets are exported as CSV automatically.
Already-downloaded files are skipped.

In [3]:
import os, requests, time
from google.colab import auth
from google.auth import default
from google.auth.transport.requests import Request as AuthRequest

auth.authenticate_user()
creds, _ = default()

FILE_IDS = ['1DanK4wGK1WNFvID3bjMKt5fszqk78yNt', '1LCedTTOCjR90C1Ia5znsUIogGG5ExjCT', '1OJjypiuk-nnW2b_Hvt3xR8GPXxaw-qdK', '1KEpFmuAKGcfSVN5vdmsTPg_RPEKO3wEU', '1yf2fkGKbcayEB9sVaa8WaKErryi2fpFT', '1eGjnagQAjLHKXrESAD49I1c_zO5YuUst', '1QaxosHWe3AvXPo5BSVQsBH6HK9bKiOOU', '1nYEzUvzVsjYwqiE9UQADDHlDezcxgc9V', '1a-RbSMwN5I7hk1zRy3UrG7LDZ4sbsLmJ', '1B5Lt-N_ds-eKBXluus3U44zKrKRP09ug', '18KEjSK8RDnPVRe5oDp2LT69wWT4ZcQcB', '1sXposXQEgDNNv3FmRh6CYvgVOOQGU3Zn', '15PJuIXzi_n0Y_vFW-Zvk6jc6FKeHmKQ5', '1tNfMJdFbb3YPKZZvBiPjcVIdz1xmChG0', '13uR1ZUisp4XMc1BPfdk2JcdWWwRUS1Wz', '1FZdgGFz7ZJJScuLEGiW79yaaSH7jTh7y', '1t1Xf-aa8sYE9UQ9mJzNyx7_ZzIFzeKYl', '1qmYCdqfqEtHfinOSdVcatczz3PcuFGS4', '1YIWP0cCN_PITCO0dD5ADJyVsFF72hztl', '1OJMjVxCMFf9_jcw82aWxfP1fe57yOZ_K', '1LiFrwuY1uwP3oAzPgbWRL6msJt9Kpax1', '1T2czVs5g8QU7SyNypppizJjw0Tov_2EQ', '17KY9Oko5Ihi0FIz0jE2fTE0V55vIch1H', '1bIiTANWie_GkQysoEzm0U8VIl1gW3rZd', '1qiILYbwtgly9o2GyWz9l5b12ZKiU_33X', '1dUtCSjTIZHaavQLSAuHoh7b1T3tfeOWl', '1iBAJGiUrSGpVnnqf2Ue-PKZ3J3tjTRtT', '1mUeQ04TiBv1veHtRcvThqMtjTm7AVjfY', '1gm1Sr_Ve_hQLu9CRYSJb68WEbbKYZVVr', '1dzTi9CXv7x1orjYmlKq7RdDko2xDrXrt', '1m4lkPouB_gPGyQZ7c4lw01GodsJtWieX', '1Ud-BjOjN7C6qBwPCU8G7X_n6ZB1jAyGa', '1_-0KVS6V0oaxf3wNl_0HP7lLbkHN3cfe', '1P2kZnY5NHsiIaoJuddxlqtezzt5gduod', '159tnyv7RG2_krA81U43vKIpgr_cukrn4', '1C43R8ry_gE5majMWAZDHXNgVGnHO1N-i', '1MIsqPQSBTnhIhIX9zrjPoQo61WFUYkRq', '1Pzc190PE5uMjCPyIkiKKQ7xEYSI8trI5', '1RXMHrMlr0eOGtVwuQ0Q2y0xX3ygyFhWZ', '1hDaRBOcTbwsU_gUYsCorDa1Ko7WiTQ6S', '1uIVvZTsVKbbZMflsbfwGFTMMOCc9sGu1', '1pw-WrE-i596G84o2mVocdKsNbS0L97NO', '1oGnuaH1Xxyr7ItgO8n0QEIahcQcUfXE7', '1jUvY_xyVt-G-DtM1O5EK--tzIv_-oiuZ', '17LpQHx_Gsa1-4VviNnTHLND8NAegkTEP', '16m6hO4n4iHwYwhkvA0sGBp27JWF2VxDY', '1B3T8UY9mXqGt22p1A-ZKDni5kttMpJu1', '1KUV9xhGMGPrjlJOOnivc0k6jXJrpmhdp', '1PQv5EiXoN6JyYixMUv8rZtZ0wFFQZsb2', '109hPBjo2uZInkzenin_CTDV_G4W84wCW', '1Y3Rv_1M0Ju2o73BovIdhkw3a8fH2heHp', '1u7CG9mbbvRdDoVcBpcH5F3DCqfNHocJc', '13LlTUHDRGwG5oeSAIC7NuBciG0n2Bknm', '1ag2tynUonpkBeg3I-5CwhqBJFZCDVz56', '1EjJ2fpawPEvcFEOUQJwOHa1tVyc8dVw1', '1IHzfIbV9FdRHJvypW1udZh8dk5NQSUa6', '1m3Cl98HIw_jfi5-Fmthd-cPAIOleUIZO', '1_S6dw8griZMAVBYPhs7lhiL5qM947JVG', '1ChG2nlaYaDrynxsNmdK0MHHeKufoLbVS', '1NufSfJyKSc0tlK6lzzx59B9H9xyXio4_', '1JFBkOcU4igbfHOxJ7X4jZikCuiYTJ-KR', '1CjrIw4rgcaOOIjSuBEb2FBCoPNiZA0Pt', '1KmxtJVfHXicLclIO0nq-3zrUKVBUimp8', '1B5qCRQeo3iAJyFm2Ov2nuYK2xi4w2k2_', '1i6XHMYS0eC-pcJ-vZNYD-vzFjD09wMUG', '1BjrufqYROqkRAc5PclKkm2zFRcUY5pEP', '1lpEmd_J-AR8DZsp4jsNYwYazQrF1uSRq', '12pEJ1YhkxJguZl0FFaKV0JXgmm9wc90Q', '10aZF7jy2dhk-xxr9q2dFtSbiM4hj29zX', '1gFoVVXA2JcMZnJzVuyUOm-RvZgrLzKQV', '185D7XIVFa3pwzpLRFKyjqu2iDxCiI_s7', '15mCD3g4v5KypgNxX_KtIPdwY7SQPA_Gj', '1f32KNJsoqyuSrzWrciEqljDXdeICI3p1', '1flTp5S5wXiR4taP-DxGndgw-gHhF2zwq', '1ipxgwZ5tyNQvmJfktWRgmheS69_HNdze', '1zeBq9yXwnKKcUF2Bs2RlkynqZxIouAHl', '1pQCWe0ou1zBMxwlELmgwVWL--SneB1DY', '15DVBEOfwxiAoVXxs42CO3kUBpoUZhFT7', '1BVik9sSjjiKpzy26AieHrRhmB8xgf4Oy', '1IiH3F0seie0JYqbjAUeXCqkrlK3isoAG', '1H8Hxxc1j2ouBsqm0NoeUhBhU7Zq4oNJa', '1kFiUUcoAjriDQ0qggMvDCs8EA-gP520R', '1cL17XHpJjScPQ0pFltblFmkN-i4xs4hx', '1logvnjVeGdj3Z5nqqcthjIqSxUqXatYG', '1kD_TMhWqTIVOhBgTjpi1mskCw8TG3_I7', '1AVQITlXonPB8-t2ywWwEPxSp2mKKQwep', '1ieE-8aW-Y55hAySKOp2gA6LcVmCK9br9', '1r7e8Dr7reQtYgZVjdwjbkTb3xVEj7Trw', '1cVWBeeysA443bAwCTfHW8lnZqWZ5PVst', '1BG9VYUZ1JbQn-UX8x-ywjEyC0aqDOGwM', '19DPasagqf9F_glFi-VDWqBZpkJoUFHxI', '1QmEKCkaxwZzP4ctes8eMuZZVmIbiuPmU', '1jWqfmUriRCqfLrTHRTucQlVD6zRuoXrF', '16idepjnxJu0xm7O9QYmtwCyncgkn4-AD', '14qbghuT2kD8WtzYFJqlD31CKIPj7bado', '1akAbRdJTqxB9PjIqbs-tt-JUyMJDYAU7', '1ra6ga68HBsCriAHjyQF6zSfr_S99z6tv', '1ueobd8iO_L4OF-SFL1dWx2H4aSmR6u8R', '1DfUheMeyb_AxqrTtGFft6_-upmnKKoT9', '1annouenLhrP60zPLdpYmnaDEvxGG8x-X']

# Google Sheets export URL (works for both Sheets and CSV files in Drive)
def get_download_url(file_id):
    # Try Sheet export first, fall back to direct download
    return f'https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv'

def get_direct_url(file_id):
    return f'https://drive.google.com/uc?id={file_id}&export=download'

def get_filename_from_response(resp, file_id):
    cd = resp.headers.get('Content-Disposition', '')
    if 'filename=' in cd:
        fname = cd.split('filename=')[-1].strip().strip('"').strip("'")
        # Clean up any export suffix added by Sheets
        fname = fname.replace(' - Sheet1', '').replace('.csv.csv', '.csv')
        if not fname.endswith('.csv'):
            fname += '.csv'
        return fname
    return f'{file_id}.csv'

print(f'Downloading {len(FILE_IDS)} files to:')
print(f'  {DEST_SUB}')
print()

# Refresh credentials
creds.refresh(AuthRequest())
headers = {'Authorization': f'Bearer {creds.token}'}

ok, skipped, failed = 0, [], []

for i, fid in enumerate(FILE_IDS):
    # Try Sheets export URL first
    for url in [
        f'https://docs.google.com/spreadsheets/d/{fid}/export?format=csv',
        f'https://drive.google.com/uc?id={fid}&export=download',
    ]:
        try:
            resp = requests.get(url, headers=headers, allow_redirects=True, timeout=60)
            if resp.status_code == 200 and len(resp.content) > 100:
                # Reject HTML responses (login redirects)
                content_start = resp.content[:50].decode('utf-8', errors='ignore').lower()
                if '<!doctype' in content_start or '<html' in content_start:
                    continue   # try next URL
                fname = get_filename_from_response(resp, fid)
                dest  = os.path.join(DEST_SUB, fname)
                # Skip if already exists with valid content
                if os.path.exists(dest) and os.path.getsize(dest) > 500:
                    # Verify existing file is not HTML
                    with open(dest, 'r', errors='ignore') as fch:
                        if '<!doctype' not in fch.read(50).lower():
                            skipped.append(fname)
                            print(f'  [{i+1:3d}/{len(FILE_IDS)}] Skip: {fname}', end='\r')
                            break
                with open(dest, 'wb') as f:
                    f.write(resp.content)
                ok += 1
                sz = len(resp.content)/1024
                print(f'  [{i+1:3d}/{len(FILE_IDS)}] OK ({sz:.0f} KB): {fname}')
                break
        except Exception as e:
            continue
    else:
        failed.append(fid)
        print(f'  [{i+1:3d}/{len(FILE_IDS)}] FAILED: {fid}')
    time.sleep(0.2)

print(f'\nDone: {ok} downloaded, {len(skipped)} skipped, {len(failed)} failed')
if failed:
    print(f'Failed IDs: {failed}')


  /content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6

  [  1/100] FAILED: 1DanK4wGK1WNFvID3bjMKt5fszqk78yNt
  [  2/100] FAILED: 1LCedTTOCjR90C1Ia5znsUIogGG5ExjCT
  [  3/100] FAILED: 1OJjypiuk-nnW2b_Hvt3xR8GPXxaw-qdK
  [  4/100] FAILED: 1KEpFmuAKGcfSVN5vdmsTPg_RPEKO3wEU
  [  5/100] FAILED: 1yf2fkGKbcayEB9sVaa8WaKErryi2fpFT
  [  6/100] FAILED: 1eGjnagQAjLHKXrESAD49I1c_zO5YuUst
  [  7/100] FAILED: 1QaxosHWe3AvXPo5BSVQsBH6HK9bKiOOU
  [  8/100] FAILED: 1nYEzUvzVsjYwqiE9UQADDHlDezcxgc9V
  [  9/100] FAILED: 1a-RbSMwN5I7hk1zRy3UrG7LDZ4sbsLmJ
  [ 10/100] FAILED: 1B5Lt-N_ds-eKBXluus3U44zKrKRP09ug
  [ 11/100] FAILED: 18KEjSK8RDnPVRe5oDp2LT69wWT4ZcQcB
  [ 12/100] FAILED: 1sXposXQEgDNNv3FmRh6CYvgVOOQGU3Zn
  [ 13/100] FAILED: 15PJuIXzi_n0Y_vFW-Zvk6jc6FKeHmKQ5
  [ 14/100] FAILED: 1tNfMJdFbb3YPKZZvBiPjcVIdz1xmChG0
  [ 15/100] FAILED: 13uR1ZUisp4XMc1BPfdk2JcdWWwRUS1Wz
  [ 16/100] FAILED: 1FZdgGFz7ZJJScuLEGiW79yaaSH7jTh7y
  [ 17/100] FAILED: 1t1Xf-aa8sYE9UQ9mJzNyx7_Z

Cell 2 — Detect chromosome/arm and rename
Reads first 100 rows of each file to detect chromosome and arm (p/q). Uses centromere position estimates to distinguish p from q arm.

In [4]:
# ============================================================
# Rename downloaded files from ID-based names to proper V6 names
# Each file contains DMB data — read first column to detect chrom
# ============================================================

import os
import pandas as pd

DEST = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'

# Find all ID-named CSV files (they match: no underscore pattern of V6 files)
id_files = [f for f in os.listdir(DEST)
            if f.endswith('.csv')
            and '_V6_dmb_' not in f
            and '_V6_ct_' not in f
            and len(f.split('.')[0]) > 20]  # Drive IDs are long

print(f'Found {len(id_files)} ID-named files to process')
print()

renamed, failed, skipped = [], [], []

for fname in sorted(id_files):
    fpath = os.path.join(DEST, fname)
    try:
        # Read just the first few rows to detect content
        df = pd.read_csv(fpath, nrows=5, on_bad_lines='skip')

        if len(df) == 0 or len(df.columns) < 3:
            failed.append((fname, 'empty or too few columns'))
            continue

        cols = [c.lower() for c in df.columns]

        # Detect chromosome from 'chrom' column or from data values
        chrom = None
        if 'chrom' in cols:
            chrom_col = df.columns[cols.index('chrom')]
            chrom = df[chrom_col].iloc[0]
            if not str(chrom).startswith('chr'):
                chrom = 'chr' + str(chrom)
        elif 'chr' in cols:
            chrom_col = df.columns[cols.index('chr')]
            chrom = df[chrom_col].iloc[0]
            if not str(chrom).startswith('chr'):
                chrom = 'chr' + str(chrom)

        if chrom is None:
            failed.append((fname, f'no chrom column found. Cols: {list(df.columns[:5])}'))
            continue

        # Detect arm (p or q) from filename hint or mid_mb position
        # p-arm = smaller coordinates (mid_mb < chromosome centromere)
        # q-arm = larger coordinates
        # Use file content if 'arm' column exists, else use median position
        arm = None
        if 'arm' in cols:
            arm_col = df.columns[cols.index('arm')]
            arm = str(df[arm_col].iloc[0]).lower()
        else:
            # Read more rows to estimate arm by median position
            df_full = pd.read_csv(fpath, nrows=100, on_bad_lines='skip')
            if 'mid_mb' in [c.lower() for c in df_full.columns]:
                mid_col = df_full.columns[[c.lower() for c in df_full.columns].index('mid_mb')]
                median_pos = df_full[mid_col].median()
                # Rough centromere positions in Mb for each chromosome
                centromeres = {
                    'chr1':125,'chr2':93,'chr3':91,'chr4':50,'chr5':48,
                    'chr6':61,'chr7':59,'chr8':45,'chr9':49,'chr10':40,
                    'chr11':53,'chr12':35,'chr13':17,'chr14':19,'chr15':20,
                    'chr16':36,'chr17':24,'chr18':17,'chr19':28,'chr20':27,
                    'chr21':12,'chr22':15,'chrX':61,'chrY':12
                }
                cen = centromeres.get(str(chrom), 60)
                arm = 'p' if median_pos < cen else 'q'
            else:
                # Fall back: assume q-arm (p-arms for most chroms are small/empty)
                arm = 'q'

        # Build the correct filename
        new_name = f'{chrom}_V6_dmb_{arm}.csv'
        new_path = os.path.join(DEST, new_name)

        # Check if destination already exists with content
        if os.path.exists(new_path) and os.path.getsize(new_path) > 1000:
            # Compare sizes — if new file is larger, overwrite; else skip
            existing_sz = os.path.getsize(new_path)
            new_sz      = os.path.getsize(fpath)
            if new_sz <= existing_sz:
                skipped.append(f'{fname} -> {new_name} (kept existing, {existing_sz//1024} KB)')
                os.remove(fpath)  # remove the ID-named duplicate
                continue

        os.rename(fpath, new_path)
        renamed.append(f'{fname} -> {new_name}')
        print(f'  {fname[:20]}... -> {new_name}')

    except Exception as e:
        failed.append((fname, str(e)[:100]))
        print(f'  FAIL {fname[:20]}...: {e}')

print(f'\nRenamed : {len(renamed)}')
print(f'Skipped : {len(skipped)}')
print(f'Failed  : {len(failed)}')
if failed:
    print('Failed files:')
    for f, e in failed[:5]:
        print(f'  {f}: {e}')

# Final count
all_dmb = [f for f in os.listdir(DEST) if '_V6_dmb_' in f and f.endswith('.csv')]
print(f'\nV6 DMB files now in folder: {len(all_dmb)}')
chroms = sorted(set(f.split("_")[0] for f in all_dmb))
print(f'Chromosomes: {chroms}')


Found 0 ID-named files to process


Renamed : 0
Skipped : 0
Failed  : 0

V6 DMB files now in folder: 48
Chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']


Cell 3 — Verify renamed files

In [5]:
# Verify arm detection was correct — show first few rows of each renamed file
import os, pandas as pd

DEST = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'

dmb_files = sorted([f for f in os.listdir(DEST) if '_V6_dmb_' in f and f.endswith('.csv')])
print(f'Total DMB files: {len(dmb_files)}')
print()

for fname in dmb_files[:15]:
    fpath = os.path.join(DEST, fname)
    sz = os.path.getsize(fpath)/1024
    try:
        df = pd.read_csv(fpath, nrows=2, on_bad_lines='skip')
        cols = list(df.columns[:4])
        nrows_full = sum(1 for _ in open(fpath)) - 1
        print(f'  {fname:<35} {sz:>6.0f} KB  rows={nrows_full}  cols={cols}')
    except Exception as e:
        print(f'  {fname:<35} ERROR: {e}')


Total DMB files: 48

  chr10_V6_dmb_p.csv                     457 KB  rows=3273  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr10_V6_dmb_q.csv                     996 KB  rows=7026  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr11_V6_dmb_p.csv                     513 KB  rows=3682  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr11_V6_dmb_q.csv                     893 KB  rows=6299  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr12_V6_dmb_p.csv                     361 KB  rows=2592  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr12_V6_dmb_q.csv                    1039 KB  rows=7322  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr13_V6_dmb_p.csv                  ERROR: No columns to parse from file
  chr13_V6_dmb_q.csv                     878 KB  rows=6226  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr14_V6_dmb_p.csv                  ERROR: No columns to parse from file
  chr14_V6_dmb_q.csv                     934 KB  rows=6624  cols=['start', 'end', 'mid_mb', 'h1_beta']
  chr

## Step 2 — Verify downloaded files

In [6]:
import os

ROOT = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6'
SUB  = ROOT + '/Methylation Paper cpG_v6'

for label, d in [('Root (chr20-22/X/Y)', ROOT), ('Subfolder (chr1-19)', SUB)]:
    files = os.listdir(d)
    dmb   = sorted([f for f in files if '_V6_dmb_' in f and f.endswith('.csv')])
    ct    = sorted([f for f in files if '_V6_ct_'  in f and f.endswith('.csv')])
    chroms = sorted(set(f.split('_')[0] for f in dmb))
    print(f'{label}:')
    print(f'  DMB files : {len(dmb)}  CT files: {len(ct)}')
    print(f'  Chroms    : {chroms}')
    print()

CHROMS = [f'chr{i}' for i in list(range(1,23))+['X','Y']]
all_dmb = []
for d in [ROOT, SUB]:
    all_dmb += [f for f in os.listdir(d) if '_V6_dmb_' in f]
have = set(f.split('_')[0] for f in all_dmb)
missing = [c for c in CHROMS if c not in have]
print(f'Missing chromosomes: {missing if missing else "None — all present!"}')


Root (chr20-22/X/Y):
  DMB files : 10  CT files: 5
  Chroms    : ['chr20', 'chr21', 'chr22', 'chrX', 'chrY']

Subfolder (chr1-19):
  DMB files : 48  CT files: 24
  Chroms    : ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']

Missing chromosomes: None — all present!


In [7]:
import os, pandas as pd

DEST = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'

all_files = sorted(os.listdir(DEST))
print(f'Total files in folder: {len(all_files)}')
print()

# Separate by type
v6_dmb   = [f for f in all_files if '_V6_dmb_' in f]
id_named = [f for f in all_files if len(f.split('.')[0]) > 20 and f.endswith('.csv')]
other    = [f for f in all_files if f not in v6_dmb and f not in id_named]

print(f'Properly named V6 DMB files: {len(v6_dmb)}')
for f in sorted(v6_dmb)[:6]:
    sz = os.path.getsize(os.path.join(DEST,f))//1024
    print(f'  {f}  ({sz} KB)')

print(f'\nID-named files (still need renaming): {len(id_named)}')
for f in sorted(id_named)[:6]:
    sz = os.path.getsize(os.path.join(DEST,f))//1024
    print(f'  {f}  ({sz} KB)')

print(f'\nOther files: {len(other)}')
for f in other[:6]:
    print(f'  {f}')

# Peek at first ID-named file to see what's inside
if id_named:
    sample = os.path.join(DEST, id_named[0])
    print(f'\nPeeking inside: {id_named[0]}')
    try:
        df = pd.read_csv(sample, nrows=3)
        print(f'  Columns: {list(df.columns)}')
        print(f'  First row: {df.iloc[0].to_dict()}')
    except Exception as e:
        # Try reading as plain text
        with open(sample, 'r', errors='replace') as f_:
            for i, line in enumerate(f_):
                if i < 5: print(f'  L{i}: {line[:120].rstrip()}')
                else: break

Total files in folder: 172

Properly named V6 DMB files: 48
  chr10_V6_dmb_p.csv  (457 KB)
  chr10_V6_dmb_q.csv  (995 KB)
  chr11_V6_dmb_p.csv  (513 KB)
  chr11_V6_dmb_q.csv  (892 KB)
  chr12_V6_dmb_p.csv  (361 KB)
  chr12_V6_dmb_q.csv  (1039 KB)

ID-named files (still need renaming): 0

Other files: 124
  GSM429321_H1_1.wig
  GSM429322_H1_2.wig
  GSM432687_IMR90_1.wig
  GSM432688_IMR90_2.wig
  chr10_V6_ct_scores.csv
  chr10_V6_metrics.json


In [8]:
import os

DEST = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'

# Remove (1) duplicate files — these are exact copies
dupes = [f for f in os.listdir(DEST) if ' (1)' in f or ' (2)' in f]
print(f'Removing {len(dupes)} duplicate files...')
for f in dupes:
    os.remove(os.path.join(DEST, f))

# Remove corrupt HTML files (ID-named, ~879 KB)
corrupt = [f for f in os.listdir(DEST)
           if len(f.split('.')[0]) > 20 and f.endswith('.csv')]
print(f'Removing {len(corrupt)} corrupt HTML files...')
for f in corrupt:
    os.remove(os.path.join(DEST, f))

remaining = [f for f in os.listdir(DEST) if '_V6_dmb_' in f]
print(f'Clean V6 DMB files remaining: {len(remaining)}')
chroms = sorted(set(f.split('_')[0] for f in remaining))
print(f'Chromosomes: {chroms}')

Removing 0 duplicate files...
Removing 0 corrupt HTML files...
Clean V6 DMB files remaining: 48
Chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']


## Step 2b — Cleanup corrupt and duplicate files
Removes HTML redirect files and `(1)` duplicates before loading DMBs.

In [9]:
# ============================================================
# Cleanup — remove corrupt HTML files and duplicate (1) files
# Run this before loading DMBs to ensure clean file set
# ============================================================
import os

DEST = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'

# Remove (1) and (2) duplicate files
dupes = [f for f in os.listdir(DEST)
         if (' (1)' in f or ' (2)' in f or ' (3)' in f) and f.endswith('.csv')]
print(f'Removing {len(dupes)} duplicate files...')
for f in dupes:
    os.remove(os.path.join(DEST, f))

# Remove corrupt HTML files (ID-named ~879 KB files)
corrupt = [f for f in os.listdir(DEST)
           if len(f.split('.')[0]) > 20 and f.endswith('.csv')]
print(f'Removing {len(corrupt)} corrupt HTML files...')
for f in corrupt:
    fpath = os.path.join(DEST, f)
    # Verify it's actually HTML before deleting
    with open(fpath, 'r', errors='replace') as fh:
        first = fh.read(50)
    if '<!doctype' in first.lower() or '<html' in first.lower():
        os.remove(fpath)
    else:
        print(f'  Kept (not HTML): {f}')

# Final count
dmb_files = sorted([f for f in os.listdir(DEST) if '_V6_dmb_' in f])
print(f'\nClean V6 DMB files: {len(dmb_files)}')
chroms = sorted(set(f.split("_")[0] for f in dmb_files))
p_arms = [f for f in dmb_files if '_dmb_p' in f]
q_arms = [f for f in dmb_files if '_dmb_q' in f]
print(f'p-arm files: {len(p_arms)}  q-arm files: {len(q_arms)}')
print(f'Chromosomes: {chroms}')


Removing 0 duplicate files...
Removing 0 corrupt HTML files...

Clean V6 DMB files: 48
p-arm files: 24  q-arm files: 24
Chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']


## Step 3 — Load all V6 DMBs

In [10]:
import numpy as np
import pandas as pd

CHROMS = [f'chr{i}' for i in list(range(1,23))+['X','Y']]
ROOT = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6'
SUB  = ROOT + '/Methylation Paper cpG_v6'

print('Loading V6 DMBs...')
all_dmbs, seen = [], set()

for search_dir in [ROOT, SUB]:
    for fname in sorted(os.listdir(search_dir)):
        if '_V6_dmb_' not in fname or not fname.endswith('.csv') or fname in seen:
            continue
        seen.add(fname)
        fpath = os.path.join(search_dir, fname)
        try:
            if os.path.getsize(fpath) <= 1: continue
            df = pd.read_csv(fpath)
            if len(df) == 0: continue
            chrom = fname.split('_')[0]
            df['chrom'] = chrom
            if 'abs_delta' not in df.columns and 'delta' in df.columns:
                df['abs_delta'] = df['delta'].abs()
            if 'ram_sig_99' not in df.columns:
                df['ram_sig_99'] = df['abs_delta'] > 0.20
            if 'start' not in df.columns and 'mid_mb' in df.columns:
                df['start'] = (df['mid_mb']*1e6).astype(int)
                df['end']   = df['start'] + WINDOW_SIZE
            sig = df[df['ram_sig_99']]
            if len(sig): all_dmbs.append(sig)
        except (pd.errors.EmptyDataError, pd.errors.ParserError): pass
        except Exception as e: print(f'  Skip {fname}: {e}')

dmb_df = pd.concat(all_dmbs, ignore_index=True)
print(f'V6 DMBs (significant): {len(dmb_df):,}')
print(f'Chromosomes          : {sorted(dmb_df.chrom.unique())}')


Loading V6 DMBs...
V6 DMBs (significant): 77,066
Chromosomes          : ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX']


In [11]:
import os

DEST = '/content/drive/MyDrive/Meth3DNet_TCGA/Methylation_Paper_CpG_v6/Methylation Paper cpG_v6'

# Remove (1) duplicate files — these are exact copies
dupes = [f for f in os.listdir(DEST) if ' (1)' in f or ' (2)' in f]
print(f'Removing {len(dupes)} duplicate files...')
for f in dupes:
    os.remove(os.path.join(DEST, f))

# Remove corrupt HTML files (ID-named, ~879 KB)
corrupt = [f for f in os.listdir(DEST)
           if len(f.split('.')[0]) > 20 and f.endswith('.csv')]
print(f'Removing {len(corrupt)} corrupt HTML files...')
for f in corrupt:
    os.remove(os.path.join(DEST, f))

remaining = [f for f in os.listdir(DEST) if '_V6_dmb_' in f]
print(f'Clean V6 DMB files remaining: {len(remaining)}')
chroms = sorted(set(f.split('_')[0] for f in remaining))
print(f'Chromosomes: {chroms}')

Removing 0 duplicate files...
Removing 0 corrupt HTML files...
Clean V6 DMB files remaining: 48
Chromosomes: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']


## Step 4 — Build probe set from Illumina 450K manifest

In [12]:
print('Building probe set from manifest...')
TARGET_PROBES = set()

header_row = 0
with open(MANIFEST, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if line.startswith('IlmnID') or line.startswith('Name'):
            header_row = i; break
print(f'Manifest header at row {header_row}')

man = pd.read_csv(MANIFEST, skiprows=header_row, low_memory=False)
name_col = next((c for c in man.columns if c in ['Name','IlmnID']), man.columns[0])
chr_col  = next((c for c in man.columns if c in ['CHR','Chr','Chromosome']), None)
pos_col  = next((c for c in man.columns if c in ['MAPINFO','MapInfo','Position']), None)
print(f'Manifest: {man.shape}  probe={name_col} chr={chr_col} pos={pos_col}')

man = man[[name_col,chr_col,pos_col]].dropna()
man.columns = ['probe_id','chrom','position']
man['chrom']    = 'chr' + man['chrom'].astype(str).str.replace('chr','',regex=False)
man['position'] = pd.to_numeric(man['position'], errors='coerce')
man = man.dropna(subset=['position'])
man['position'] = man['position'].astype(int)
print(f'Valid probes: {len(man):,}')

print('Intersecting with V6 DMB windows...')
for chrom in CHROMS:
    intervals = dmb_df[dmb_df['chrom']==chrom].sort_values('start')
    probes    = man[man['chrom']==chrom]
    if len(intervals)==0 or len(probes)==0: continue
    starts = intervals['start'].values
    ends   = intervals['end'].values
    for _, row in probes.iterrows():
        pos = row['position']
        idx = np.searchsorted(starts, pos, side='right') - 1
        if idx >= 0 and pos <= ends[idx]:
            TARGET_PROBES.add(row['probe_id'])

print(f'Probes overlapping V6 DMBs: {len(TARGET_PROBES):,}')


Building probe set from manifest...
Manifest header at row 7
Manifest: (486428, 33)  probe=IlmnID chr=CHR pos=MAPINFO
Valid probes: 485,512
Intersecting with V6 DMB windows...
Probes overlapping V6 DMBs: 134,178


## Step 5 — Scan TCGA methylation file
**Takes 20–60 minutes.** Progress every 100,000 lines. Do not interrupt.

In [ ]:
print('Scanning TCGA file (low-RAM mode — writes directly to disk)...')
print(f'File: {METH_FILE}')
print(f'Size: {os.path.getsize(METH_FILE)/1e9:.1f} GB')

total_lines = 0
found_count = 0

with open(METH_FILE, 'rt', errors='replace') as fh, \
     open(OUTPUT_FILE, 'wt', encoding='utf-8') as out:

    header = fh.readline().rstrip('\n')
    sep = '\t' if '\t' in header else ','
    sample_ids = [s.strip()[:15] for s in header.split(sep)[1:]]
    n_cols = len(sample_ids)
    print(f'Samples: {n_cols:,}  Target probes: {len(TARGET_PROBES):,}')

    # Write output header immediately
    out.write('probe_id\t' + '\t'.join(sample_ids) + '\n')

    for line in fh:
        total_lines += 1
        if total_lines % 100_000 == 0:
            pct = found_count / max(len(TARGET_PROBES), 1) * 100
            print(f'  {total_lines:,} lines | {found_count:,} probes ({pct:.0f}%)',
                  end='\r', flush=True)

        parts    = line.rstrip('\n').split(sep)
        probe_id = parts[0].strip().strip('"')

        if not TARGET_PROBES or probe_id in TARGET_PROBES:
            vals    = parts[1:]
            trimmed = vals[:n_cols] + ['NA'] * (n_cols - len(vals[:n_cols]))
            out.write(probe_id + '\t' + '\t'.join(trimmed) + '\n')
            found_count += 1

        if TARGET_PROBES and found_count >= len(TARGET_PROBES):
            print(f'\n  All target probes found — stopping early.')
            break

print(f'\nScanned: {total_lines:,} lines')
print(f'Probes written: {found_count:,}')
print(f'Output: {OUTPUT_FILE}')
print(f'Size: {os.path.getsize(OUTPUT_FILE)/1e6:.0f} MB')

Scanning TCGA file (low-RAM mode — writes directly to disk)...
File: /content/drive/MyDrive/Meth3DNet_TCGA/jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv
Size: 49.2 GB
Samples: 9,854  Target probes: 134,178


## Step 6 — Write output

In [ ]:
# Cell 19 already wrote directly to disk in low-RAM mode.
# This cell just verifies the output exists.
import os
if os.path.exists(OUTPUT_FILE):
    sz = os.path.getsize(OUTPUT_FILE)/1e6
    print(f'Output file confirmed: {OUTPUT_FILE}')
    print(f'Size   : {sz:.0f} MB')
    print(f'Probes : {found_count:,}')
    print(f'Samples: {n_cols:,}')
    print()
    print('Next: upload tcga_dmb_probes_full.tsv to Kaggle as dataset tcga-pancan-methylation')
else:
    print('Output file not found. Re-run Cell 19.')


## Step 7 — Preview output

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT_FILE, sep='\t', index_col=0, nrows=5)
print(f'Shape preview: {df.shape}')
print(df.iloc[:, :4])
